Notes:
- Transformer tokens in language, predicts next token
- We convert language to a sequence of integers with a tokenizer
- We convert integers to vectors with a lookup table
- output is a cevtor of logits, convert to a probability distn with a softmax, and can then convert this to a token.


The most basic object in Hugging Face transformers library is the `pipeline()` function. It connects a model with its necessary preprocessing and postprocessing steps, allowing us to directly input any text and get an intelligible answer:

In [18]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis")
classifier("I've been waiting for a HuggingFace Course my whole life")

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 14198.08it/s]


[{'label': 'POSITIVE', 'score': 0.9516069293022156}]

## Zero-shot classification

Sometimes we need to classify texts that haven't been labelled. `zero-shot classification` pipeline is very powerful, allows you to specify which labels to use for the classification.

In [19]:
from transformers import pipeline

classifier = pipeline("zero-shot-classification")
classifier(
    "This is a course about the Transformers library",
    candidate_labels = ["education", "politics", "business"],
)

Loading weights: 100%|██████████| 515/515 [00:00<00:00, 3709.26it/s]


{'sequence': 'This is a course about the Transformers library',
 'labels': ['education', 'business', 'politics'],
 'scores': [0.8445951342582703, 0.1119769886136055, 0.043427880853414536]}

## Text generation

The main idea here is that you provide a prompt and the model will auto-complete it by generating the remaining text. This is similar to the predictive text feature that is found on many phones. Text generation involves randomness, so it’s normal if you don’t get the same results as shown below.

In [22]:
generator = pipeline("text-generation")
generator("In this course, we will teach you how to")

Loading weights: 100%|██████████| 326/326 [00:00<00:00, 3324.30it/s]


[{'generated_text': "In this course, we will teach you how to use Python to automate tasks on your computer. You'll learn how to use Python's built-in libraries, such as os and subprocess, to interact with your operating system and run commands. We'll also show you how to use the argparse library to parse command-line arguments and create a simple command-line interface.\n\nWe'll start by going over the basics of Python and the Python interpreter. Then, we'll dive into the os and subprocess libraries and show you how to use them to interact with your operating system. We'll also cover the argparse library and show you how to use it to parse command-line arguments.\n\nAfter that, we'll go over some examples of how to use Python to automate tasks on your computer. This will include tasks such as running shell commands, checking the status of files and directories, and creating and managing files and directories.\n\nWe'll also cover some best practices for writing Python scripts and using

## Mask filling

In [12]:
from transformer_lens.model_bridge import TransformerBridge
import torch as t

In [13]:
reference_gpt2 = TransformerBridge.boot_transformers("gpt2")

sorted_vocab = sorted(
    reference_gpt2.tokenizer.get_vocab().items(),
    key=lambda n: n[1]
)

print(len(sorted_vocab))

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 7047.97it/s]


50257


In [17]:
# print(sorted_vocab[20:])
print(sorted_vocab[1200:1300])

# print(sorted_vocab[-20:])

[('Ġchild', 1200), ('Ġsince', 1201), ('ired', 1202), ('less', 1203), ('Ġlife', 1204), ('Ġdevelop', 1205), ('ittle', 1206), ('Ġdep', 1207), ('Ġpass', 1208), ('ãĥ', 1209), ('Ġturn', 1210), ('orn', 1211), ('This', 1212), ('bers', 1213), ('ross', 1214), ('ĠAd', 1215), ('Ġfr', 1216), ('Ġresp', 1217), ('Ġsecond', 1218), ('oh', 1219), ('Ġ/', 1220), ('Ġdisc', 1221), ('Ġ&', 1222), ('Ġsomething', 1223), ('Ġcomple', 1224), ('Ġed', 1225), ('Ġfil', 1226), ('Ġmonth', 1227), ('aj', 1228), ('uc', 1229), ('Ġgovernment', 1230), ('Ġwithout', 1231), ('Ġleg', 1232), ('Ġdist', 1233), ('Ġput', 1234), ('Ġquest', 1235), ('ann', 1236), ('Ġprot', 1237), ('20', 1238), ('Ġnever', 1239), ('ience', 1240), ('Ġlevel', 1241), ('Ġart', 1242), ('Ġthings', 1243), ('Ġmight', 1244), ('Ġeffect', 1245), ('Ġcontro', 1246), ('Ġcent', 1247), ('Ġ18', 1248), ('Ġallow', 1249), ('Ġbelie', 1250), ('chool', 1251), ('ott', 1252), ('Ġincre', 1253), ('Ġfeel', 1254), ('Ġresult', 1255), ('Ġlot', 1256), ('Ġfun', 1257), ('ote', 1258), ('Ġty'

When we get to the end of the vocabulary, there will be some weird tokens, because we already have exhausted all of the short frequently occurring ones

### Tokenization annoyances
- Whether a word begins with a capital or space matters!
- Arithmetic is a mess

> Transformers in the `transformer_lens` library has a `to_tokens` method that converts text to tokens.

In [5]:
print(reference_gpt2.to_str_tokens("Kane", prepend_bos=False))
print(reference_gpt2.to_str_tokens(" Kane", prepend_bos=False))
print(reference_gpt2.to_str_tokens("kane", prepend_bos=False))

print(reference_gpt2.to_str_tokens("56873+3184623=123456789-1000000000"))

['K', 'ane']
[' Kane']
['k', 'ane']
['<|endoftext|>', '568', '73', '+', '318', '46', '23', '=', '123', '45', '67', '89', '-', '1', '000000', '000']


# Text generation
## Step 1: Convert text to tokens
The sequence gets tokenize, so it has shape [batch, seq_len].

In [6]:
reference_text = "I love Machine Learning"
tokens = reference_gpt2.to_tokens(reference_text)

print(tokens)
# print(tokens.shape)


tensor([[50256,    40,  1842, 10850, 18252]])


## Step 2: Map tokens to logits
From our input of shap `[batch, seq_len]`, we get output of shape `[batch, seq_len, vocab_size]`, 

In [7]:
logits, cache = reference_gpt2.run_with_cache(tokens)
print(logits.shape)

torch.Size([1, 5, 50257])


## Step 3: Convert the logits to a distribution with a softmax

In [8]:
probs = logits.softmax(dim = -1)
print(probs.shape)

torch.Size([1, 5, 50257])


## Step 4: Map distribution to a token

Note that we are indexing `logits[0, -1]`. This is because logits have shape `[1, sequence_length, vocab_size]`, so this indexing returns the vector of length `vocab_size` representing the model's prediction for what token follows the **last** token in the input sequence.

In [9]:
next_token = logits[0, -1].argmax(dim = -1)
next_char = reference_gpt2.to_string(next_token)
print(repr(next_char))

'.'


## Step 5: Add this to the end of the input
There are more efficient ways to do this, but does not matter right now

In [10]:
print(f"Sequence so far: {reference_gpt2.to_string(tokens)[0]!r}")

for i in range(10):
    print(f"{tokens.shape[-1] + 1}th char = {next_char!r}")
    # Define new input sequence, by appending the previously generated token
    tokens = t.cat([tokens, next_token[None, None]], dim=-1)
    # Pass our new sequence through the model, to get new output
    logits = reference_gpt2(tokens)
    # Get the predicted token at the end of our sequence
    next_token = logits[0, -1].argmax(dim=-1)
    # Decode and print the result
    next_char = reference_gpt2.to_string(next_token)

Sequence so far: '<|endoftext|>I love Machine Learning'
6th char = '.'
7th char = ' I'
8th char = ' love'
9th char = ' the'
10th char = ' fact'
11th char = ' that'
12th char = ' it'
13th char = "'s"
14th char = ' a'
15th char = ' great'


# Transformer Implementation


## Parameters and Activations
- Parameters are the weights and biases that are learned during training.
- Activations are temporary numbers calculated during a forward pass, that are functions of the input.

### Print All activation shapes of reference model

> 在 Transformer 的结构中，每一个计算节点都可能产生一个 Activation Tensor. 类似就是Forward pass 过程中产生的临时结果

In [11]:
for activation_name, activation in cache.items():
    if ".0." in activation_name or "blocks" not in activation_name:
        print(f"{activation_name:30} {tuple(activation.shape)}")

embed.hook_in                  (1, 5)
embed.hook_out                 (1, 5, 768)
hook_embed                     (1, 5, 768)
pos_embed.hook_in              (1, 5)
pos_embed.hook_out             (1, 5, 768)
hook_pos_embed                 (1, 5, 768)
blocks.0.hook_in               (1, 5, 768)
blocks.0.hook_resid_pre        (1, 5, 768)
blocks.0.ln1.hook_in           (1, 5, 768)
blocks.0.ln1.hook_scale        (1, 5, 1)
blocks.0.ln1.hook_normalized   (1, 5, 768)
blocks.0.ln1.hook_out          (1, 5, 768)
blocks.0.attn.hook_in          (1, 5, 768)
blocks.0.attn.q.hook_in        (1, 5, 768)
blocks.0.attn.q.hook_out       (1, 5, 12, 64)
blocks.0.attn.hook_q           (1, 5, 12, 64)
blocks.0.attn.k.hook_in        (1, 5, 768)
blocks.0.attn.k.hook_out       (1, 5, 12, 64)
blocks.0.attn.hook_k           (1, 5, 12, 64)
blocks.0.attn.v.hook_in        (1, 5, 768)
blocks.0.attn.v.hook_out       (1, 5, 12, 64)
blocks.0.attn.hook_v           (1, 5, 12, 64)
blocks.0.attn.hook_attn_scores (1, 12, 5, 5)
blo

### Config
The

## Tokenization & Embedding
The input tokens _t_ are intergers. We get them from taking a sequence, and tokenizing it.

Token embedding is a lookup table mapping tokens to vectors, which is implemented as a matrix $W_E$. The matrix consists of a stack of token embedding vectors.



## Attention

Attention moves information from prior positions in the sequence to the current token.

This is done for every token in parallel, the only difference is that we look backwards only. This means later tokens have more of the sequence that they can look at.

Attention layers are made up of `n_heads` heas - each with their own parameters, own attention pattern, and own information about how to copy things from source to detination. 

Each attention head is made up of three components: keys, queries, and values (K, Q and V). These names come from their analogy to retrieval systems.

